In [ ]:
!pip install openai

In [4]:
from getpass import getpass
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
#from google.colab import userdata
#OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)


In [5]:
MODEL_NAME = "gpt-4.1-mini"

In [6]:
def generate_response(messages, model = "gpt-4.1-mini" , **kwargs):
    if type(messages) == str:
        messages = [{"role": "user", "content": messages}]
    response = client.chat.completions.create(
        model = model,
        messages = messages,
        **kwargs
    )
    if kwargs.get("n", 1) > 1:
        return [choice.message.content for choice in response.choices]

    return response.choices[0].message.content


def generate_response_dense(messages, model = "gpt-4.1-mini", max_tokens = 250, temperature=0.5, **kwargs):
    answer = generate_response(messages, model = model, max_tokens = max_tokens, temperature = temperature, **kwargs)
    
    return answer

def generate_response_reasoning(messages, model = "gpt-5.1", reasoning_effort = "medium"):
    answer = generate_response(messages, model = model, reasoning_effort = reasoning_effort)
    
    return answer

# Modelos densos


In [ ]:
# ejemplo 1
prompt = "¿Cuáles son los tres principales síntomas del COVID-19? Dimelo en menos de 25 palabras."
response = generate_response_dense(prompt,max_tokens=80)
print(response)


Los tres principales síntomas del COVID-19 son fiebre, tos seca y dificultad para respirar.


In [ ]:
# ejemplo 2 -> ¿Qué piensas que sucede?
prompt = "¿Cuál es la capital de España?."
response = generate_response_dense(prompt,max_tokens=80)
print(response)


La capital de España es Madrid.


In [ ]:
# ejemplo 3
prompt = "Los animales son ..."
response = generate_response_dense(prompt)
print(response)

Los animales son seres vivos que pertenecen al reino Animalia. Se caracterizan por ser organismos multicelulares, heterótrofos (obtienen su alimento de otros organismos), y generalmente tienen la capacidad de moverse de manera activa en algún momento de su vida. Además, presentan diferentes grados de complejidad, desde organismos simples como las esponjas hasta otros muy complejos como los mamíferos. Los animales también tienen sistemas nerviosos que les permiten responder a estímulos del ambiente.


**Ejercicio**
- Dile que te responda a una pregunta en menos de 50 palabras y menos de 50 tokens. Qué sucede?
- Prueba ahora a usar los modelos de razonamiento, con reasoning effort None, "low", y "medium". Pregunta "¿Cuanto pesa más, si un kilo de paja o el doble de medio kilo de metal?Responde sólo diciendo 1 si es la paja, 2 si es el metal o 3 si igual. Solo el número" Qué diferencias ves? Mide los tiempos de respuesta y la calidad de la respuesta.

In [17]:
generate_response_reasoning(prompt, reasoning_effort='none')

'Los animales son seres vivos que:\n\n- Nacen, crecen, se alimentan, se reproducen y mueren.  \n- Generalmente pueden moverse por sí mismos.  \n- Necesitan alimentarse de otros seres vivos (no producen su propio alimento como las plantas).  \n- Responden a estímulos del entorno (luz, sonido, peligro, comida).\n\nSi me dices el contexto (definición para tarea escolar, texto creativo, explicación científica, etc.), puedo completar la frase de la forma más adecuada.'

# Variación de parámetros


**Temperatura**

La temperatura controla la aleatoriedad de la salida del modelo. Un valor más alto genera respuestas más diversas y menos coherentes, mientras que un valor más bajo produce respuestas más conservadoras y coherentes.



In [18]:
prompt = "qué recomiendas hacer cuando llueve?"
# Respuesta más conservadora y coherente (temperatura baja)
response = generate_response_dense(prompt, temperature=0.2)
print(response)

# Respuesta más diversa y menos coherente (temperatura alta)
response = generate_response_dense(prompt, temperature=1.0)
print(response)

Cuando llueve, hay muchas actividades que puedes disfrutar dependiendo de tus gustos y circunstancias. Aquí te dejo algunas recomendaciones:

1. **Leer un buen libro**: La lluvia crea un ambiente perfecto para sumergirte en una historia.
2. **Ver películas o series**: Aprovecha para ponerte al día con tus favoritos o descubrir nuevos títulos.
3. **Cocinar o hornear**: Preparar algo rico en la cocina puede ser muy reconfortante.
4. **Escuchar música o podcasts**: Relájate con tus canciones favoritas o aprende algo nuevo.
5. **Hacer manualidades o proyectos creativos**: Pintar, dibujar, tejer o cualquier actividad artística.
6. **Organizar tu espacio**: Aprovecha para ordenar tu habitación, armario o escritorio.
7. **Meditar o practicar yoga**: La lluvia puede ayudar a crear un ambiente tranquilo para la relajación.
8. **Salir a caminar con paraguas**: Si te gusta la lluvia, salir a caminar puede ser una experiencia refrescante y diferente.
9. **Juegos de mesa o videojuegos**: Perfecto p

**max_tokens**

El parámetro max_tokens limita la longitud de la respuesta generada por el modelo. Puedes ajustarlo para obtener respuestas más cortas o más largas según tus necesidades

In [ ]:
# Respuesta corta (limitada a 10 tokens)
response = generate_response_dense(prompt, max_tokens=10)
print(response)

# Respuesta más larga (limitada a 50 tokens)
response = generate_response_dense(prompt, max_tokens=50)
print(response)

Cuando llueve, hay muchas cosas que puedes hacer
Cuando llueve, hay muchas actividades que puedes disfrutar, dependiendo de tus gustos y circunstancias. Aquí te dejo algunas recomendaciones:

1. **Leer un buen libro**: La lluvia crea un ambiente perfecto para sumergirse en una historia.
2. **


**n**

In [ ]:
generate_response_dense(prompt, max_tokens=50, n=3)

['Como soy una inteligencia artificial, no puedo ir al parque ni hacer actividades, pero puedo ayudarte a pensar en cosas divertidas para hacer allí. ¿Te gusta caminar, hacer un picnic, jugar algún deporte o simplemente relajarte y leer un libro? ¡',
 'Como soy una inteligencia artificial, no puedo ir al parque ni hacer actividades, pero puedo ayudarte a imaginar cosas divertidas para hacer allí. Por ejemplo, muchas personas disfrutan caminar, hacer picnic, leer un libro, jugar con amigos o simplemente relajarse',
 'Como soy una inteligencia artificial, no puedo ir al parque ni hacer actividades físicas, pero puedo ayudarte a pensar en cosas divertidas que a la gente le gusta hacer en el parque, como pasear, hacer un picnic, jugar con amigos, leer un']

**top_p**

In [8]:
prompt = "qué te gusta hacer en el parque?"
# bajo, menos variedad en la respuesta
generate_response_dense(prompt,top_p = 0.05)

'Como soy una inteligencia artificial, no puedo ir al parque ni hacer actividades, pero puedo ayudarte a pensar en cosas divertidas para hacer allí. Por ejemplo, muchas personas disfrutan pasear, hacer picnic, leer un libro, jugar con amigos, andar en bicicleta o simplemente relajarse y disfrutar de la naturaleza. ¿Qué te gusta hacer a ti en el parque?'

In [29]:
# alto, más variedad en la respuesta
generate_response_dense(prompt,top_p = 0.95)

'Como soy una inteligencia artificial, no tengo experiencias ni gustos personales, pero puedo contarte sobre las actividades que muchas personas disfrutan hacer en el parque. Por ejemplo, pasear, hacer ejercicio, leer un libro, hacer un picnic, jugar con amigos o simplemente relajarse y disfrutar de la naturaleza. ¿Y a ti qué te gusta hacer en el parque?'

**repetition penalty**

In [ ]:
prompt = "cuentame una historia de un gato."
# penaliza la repeticion (valores entre -2 y 2)
generate_response(prompt,frequency_penalty = 1.5, max_tokens=100)

'Claro, aquí tienes una historia sobre un gato:\n\nHabía una vez un pequeño gato llamado Momo que vivía en un pintoresco pueblo rodeado de colinas verdes y flores silvestres. Momo era curioso y aventurero, siempre explorando cada rincón del vecindario. A diferencia de otros gatos que preferían quedarse dentro de sus casas, a Momo le encantaba salir al atardecer para descubrir nuevos secretos.\n\nUn día, mientras caminaba cerca del río que cruzaba el pueblo, encontró una puerta pequeña escondida entre los arbustos. La puerta tenía grabados extraños y parecía muy antigua. Sin pensarlo dos veces, Momo decidió empujarla con su patita y entró por ella.\n\nDel otro lado encontró un jardín maravilloso lleno de árboles frutales brillantes y mariposas gigantes que revoloteaban tranquilamente. En medio del jardín había un banco donde descansaba una anciana sonriente con ojos llenos de magia.\n\n– Bienvenido, pequeño explorador – dijo la anciana con voz dulce – Has encontrado el Jardín Secreto do

In [ ]:
# no penaliza la repeticion (valores entre -2 y 2)
generate_response_dense(prompt,frequency_penalty = -1.5, max_tokens=100)

Formatos

In [12]:
prompt = "Dame un usuario de ejemplo en JSON con nombre y edad."
generate_response_dense(prompt,frequency_penalty = -1.5, max_tokens=100,response_format={"type": "json_object"})

'{\n  "nombre": "Juan Pérez",\n  "edad": 30\n}'

**Ejercicio**

- Ejercicio: Pide al modelo una historia breve (ej. “cuenta una historia sobre un gato astronauta”) con distintos valores: Temperature: 0.2,  1.0, 1.5. El objetivo es comparar cómo cambia la creatividad y coherencia.
- Ejercicio Prompt: “Escribe una canción sobre el sol.” Modifica el top_p y la temperatura.  Objetivo: Comparar cómo varía la repetición de palabras o frases.

    - Pon una temperatura de 1 y modifica el top_p
    - Pon una temperatura 2 y modifica el top_p

- Que Genere un json que contenga nombre, apellido, edad y estudios de 3 personas inventadas